# Continued Pre-training (CPT) / Domain Adaptation
### Advanced Fine-Tuning Paradigms  ·  Colab T4 (16 GB) ready

> **SFT/DPO/ORPO/SimPO all teach *behaviour*** — what shape an answer should take, which of two answers is better. They all optimise a **conditional** distribution `p(y|x)` over a curated, labelled set.
> **CPT teaches *knowledge*.** It goes back to the model's original self-supervised objective — plain next-token prediction over **unlabelled raw text** — and moves the model's **unconditional prior `p(x)`** onto a new domain. It runs *before* any instruction tuning, and no other paradigm in this folder can substitute for it.

---

## 1. Deep-Dive Conceptual Roadmap & Dataset Ecosystem

### What it is (precise terminology)

- **CPT**, in the literature **DAPT — Domain-Adaptive Pretraining** (*Gururangan et al., 2020, "Don't Stop Pretraining: Adapt Language Models to Domains and Tasks"*), is the **warm-started continuation of the causal-language-modelling (CLM) objective** on a domain corpus, initialised from converged base weights instead of random init:
  $$\mathcal{L}_{\text{CPT}}(\theta) = -\,\mathbb{E}_{x \sim \mathcal{D}_{\text{domain}}}\Big[\sum_{t=1}^{|x|} \log \pi_\theta(x_t \mid x_{<t})\Big]$$
- **The loss is un-masked over every token.** There is no prompt/completion split, no `-100` label masking, no chat template, no `(instruction, response)` pair. The data contract is literally *a pile of text*.
  - SFT optimises **`p(y|x)`** with the loss masked to the completion → typically only **~20–40 % of tokens carry gradient**.
  - CPT optimises **`p(x)`** with **100 % of tokens carrying gradient**.
- Mathematically CPT is *the same MLE problem under a shifted data distribution*, which is exactly why it has two failure modes that pretraining-from-scratch does not have:
  - **Stale optimiser state / dead LR** — a released base checkpoint is the *end* of a cosine schedule (LR ≈ 0) and ships **without** optimiser moments. Naïvely resuming trains at effectively zero step size.
  - **Catastrophic forgetting** (*Kirkpatrick et al.*) — every gradient step toward the domain basin raises loss on the original distribution.
- The **canonical modern recipe** (*Ibrahim et al., 2024, "Simple and Scalable Strategies to Continually Pre-train Large Language Models"*) is three ingredients, all of which appear in Section 3:
  1. **Re-warm** the learning rate (linear warmup from 0 again),
  2. **Re-decay** it (fresh cosine schedule over the new corpus),
  3. **Replay 1–5 %** of general-distribution data mixed into the domain corpus.
- Two distinct sub-mechanisms live under "domain adaptation":
  - **Knowledge injection** — updating the FFN/attention weights where factual associations are stored.
  - **Tokenizer / vocabulary extension** — adding embedding rows for domain jargon that the base BPE merges shred into 5–8 fragments, then **mean-initialising** the new rows from the existing embedding matrix so they start in-distribution.
- Related terminology worth keeping straight: **TAPT** (task-adaptive pretraining — same objective, but on the *unlabelled text of the target task*), and **instruction pretraining** (*Cheng et al., 2024*), where raw corpora are augmented with synthetic instruction pairs to blend CPT and instruction tuning.

### One-sentence definition of the mechanics

> **CPT resumes the base model's un-masked causal-LM cross-entropy objective over a large unlabelled domain corpus, using a re-warmed-then-re-decayed learning rate and a small replay mixture of general text, to move the model's internal prior `p(x)` onto the domain distribution before any instruction tuning happens.**

### The exact engineering problem it solves

- **SFT is capacity-bounded by dataset size, by three to four orders of magnitude.** A large SFT set (100 k examples × ~500 tokens) is ≈ **5 × 10⁷ tokens**. Real domain adaptation runs at **10⁹–10¹¹ tokens**:
  | Model | Base | CPT corpus | Tokens |
  |---|---|---|---|
  | **Code Llama** | Llama 2 | code | **500 B** |
  | **Llemma** | Code Llama | Proof-Pile-2 (math) | **55 B** |
  | **Meditron-70B** | Llama 2 | clinical guidelines + PubMed | **48 B** |
  | **SaulLM-7B** | Mistral | legal corpora | **30 B** |
  You cannot close a 1000× token gap by writing more instruction pairs — the knowledge simply has nowhere to enter the weights from.
- **SFT on unknown facts actively manufactures hallucination.** Fine-tuning on Q/A pairs whose answers are *not* in the model's prior teaches the **surface form of a confident answer** without the substance (*Gekhman et al., 2024* — new-knowledge SFT examples are learned slowly and measurably increase hallucination rate). CPT installs the substance first; SFT then only has to learn the format.
- **Tokenizer fragmentation is unfixable downstream.** If `hydroxychloroquine` costs 9 tokens, every mention burns 9× the context, spreads the concept across 9 weakly-related embeddings, and wastes 9× the FLOPs. Only CPT (with vocabulary extension) can add and *train* new embedding rows.
- **Retrieval is not a substitute.** RAG conditions on text at inference time; it does not improve the model's internal representation of the domain's language, and it pays its cost on **every** call forever. CPT pays once, offline.

---

### The Human Element — Hugging Face datasets for CPT

| HF path | What it is | Why it's structured this way for CPT |
|---|---|---|
| **`epfl-llm/guidelines`** (`train`, 37,970 docs, ~865 MB text) | The **clinical-practice-guideline corpus behind Meditron-7B/70B** — full-length documents from 16 medical sources, exposed as a single `clean_text` column. | This is the reference shape of a CPT dataset: **one column of long, unstructured prose. No prompt, no completion, no labels, no schema.** Document *length* is a feature, not a nuisance — long docs pack into 1–8 k blocks with almost zero padding waste and let the model learn long-range domain discourse structure (guideline → evidence grade → recommendation), which paragraph-level chunks destroy. |
| **`MedRAG/textbooks`** (`train`, 125,847 rows) | 18 medical textbooks, pre-chunked into paragraph-sized `content` rows. | The **opposite extreme of the same contract** — same "one text column, no labels", but rows are short. It is the clearest demonstration of *why* packing exists: fed one-row-per-sequence into a 1024-token window, most of each sequence would be padding. Also the better choice when you want the domain's *definitional* register rather than its *procedural* register. |
| **`HuggingFaceFW/fineweb-edu`** (config **`sample-10BT`**, `text`) | High-quality general web/educational text, streamable at any scale. | The **replay** half of the mixture, and it is not optional. CPT's dataset is never one corpus: the anti-forgetting recipe needs a few percent of the *original* pretraining distribution, and since no open base model publishes its exact mix, a high-quality general corpus is the standard proxy. Structured identically to the domain corpus (one raw `text` column) precisely so the two can be tokenised, packed and shuffled together by the same code path. |
| **`allenai/peS2o`** *(scale-up option)* | ~40 M full-text scientific papers — the canonical large-scale science CPT corpus. | Same single-raw-text-column contract at the 10¹⁰-token scale you would actually use in production. ⚠️ It ships a **loading script**, so it needs `datasets<4.0` (`trust_remote_code=True`) or a direct parquet/`hf_hub_download` fetch — which is why this notebook trains on `epfl-llm/guidelines` instead. |

**Why every one of these is a single unlabelled text column:** CPT's objective has no notion of input vs. target — the target *is* the shifted input. Any extra structure (roles, pairs, preferences, scores) is unusable by the loss and is simply dropped. The engineering that replaces "schema design" in CPT is **corpus curation, dedup, and mixture ratios**.

> This notebook trains on **`epfl-llm/guidelines`** (domain) mixed with **`HuggingFaceFW/fineweb-edu` / `sample-10BT`** (replay), starting from the **`Qwen/Qwen2.5-0.5B` base — deliberately *not* `-Instruct`**, because CPT precedes instruction tuning.

---

## 2. Architectural Context Block

### **[Context Block]**

#### The 'Why' — the engineering and mathematical reason for this implementation

- **Knowledge lives in the prior, and only an un-masked CLM loss touches the prior.** SFT/DPO/ORPO/SimPO all differentiate a *conditional* objective; their gradients are dominated by response-formatting directions. Factual associations sit in the FFN key–value memories, and the only signal that systematically rewrites them is *predicting the domain's own text*, token after token, at scale.
- **Why the LR must be re-warmed, not just resumed.** A base checkpoint is the endpoint of a decayed schedule: the weights sit in a sharp minimum of `D_general` and the optimiser's moments are gone. Two directions of failure:
  - **Too little LR** → the update norms are too small to leave the general-text basin; loss goes down cosmetically (style adaptation) while no knowledge lands.
  - **Too much LR / no warmup** → an immediate loss spike, the moment estimates get poisoned by the first few high-variance batches, and forgetting on `D_general` becomes catastrophic instead of gradual.
  A short linear **re-warm** rebuilds the moment estimates at a safe step size; a fresh **cosine re-decay** anneals into the new basin. This is the single most-cited difference between "CPT that works" and "CPT that lobotomises the model".
- **Why replay works, mathematically.** Forgetting is the loss on `D_general` rising as you descend `D_domain`. Mixing a fraction `α` of general data makes the *actual* objective `(1-α)·L_domain + α·L_general`, which keeps a gradient component pointing back at the original distribution at all times. Empirically **α ≈ 0.01–0.05 recovers most of the retention of a full replay at ~nothing in cost** — a strikingly cheap fix for the field's most famous failure mode.
- **Why the loss is un-masked (the token-efficiency argument).** CPT is compute-bound, so wasted tokens are wasted dollars. Completion-masked SFT extracts gradient from ~20–40 % of the tokens it pays to forward; CPT extracts it from **100 %**. At a fixed FLOP budget that is a 2.5–5× difference in effective supervision.
- **Why packing is mandatory rather than an optimisation.** Padding tokens cost identical FLOPs and contribute exactly zero gradient. Concatenating documents with `eos` separators and slicing fixed `BLOCK_SIZE` windows drives token utilisation to ~**100 %**. At CPT's token counts, a 40 % padding rate is not inefficiency, it is a **40 % larger cloud bill for the same model**.
- **Why vocabulary extension needs mean-initialisation.** New embedding rows initialised randomly (or at zero) are far outside the learned embedding manifold; their first forward pass produces garbage logits and a large, badly-conditioned gradient that destabilises the whole run. Initialising each new row at the **mean of the existing embedding matrix** (optionally the mean of the token's own old sub-token embeddings) starts it in-distribution, so the LM head sees a sane vector from step 0.

#### VRAM & Compute Impact

- **Per-step memory is essentially identical to SFT at the same sequence length.** CPT is not a memory-hungry paradigm — this is the most commonly mis-stated thing about it. What explodes is **wall-clock**, because the corpus is 10³–10⁴× larger.
  - **CPT is compute-bound. SFT is data-bound.** That single reframe drives every hardware decision below.
- **FLOPs ≈ `6 · N · D`** (params × tokens, forward+backward). **LoRA/QLoRA does not reduce this** — the frozen 4-bit weights are still multiplied in both passes. PEFT buys **optimiser/gradient memory**, never CPT's dominant cost:

  | Component | Full-FT `AdamW` (fp32) | **QLoRA r=64 (this notebook)** |
  |---|---|---|
  | Base weights (0.5 B) | 2.0 GB (fp16) | **0.40 GB** (NF4 + double quant) |
  | Trainable params | 494 M | **~35 M** (all 7 linear projections × 24 layers) |
  | Grads + optimiser states | ~5.9 GB | **~0.35 GB** (`paged_adamw_8bit`) |
  | Activations @ 1024×2, checkpointed | ~1.5 GB | **~1.5 GB** *(unchanged — same FLOPs)* |
  | **Peak** | **~9–10 GB** | **~2.5–4 GB** |

- **Sequence length is the other lever:** attention compute is `O(L²)` while `sdpa`'s memory-efficient kernel keeps memory `O(L)`. On a T4, `BLOCK_SIZE=1024` at batch 2 is comfortable; 2048 needs batch 1 + gradient checkpointing; 4096+ wants Ampere and FlashAttention-2.
- **`gradient_checkpointing=True` is the standard CPT trade:** ~30–40 % slower per step, but activation memory drops from `O(layers)` to `O(√layers)` — the difference between a 1024-token block fitting and OOM.
- **Honest throughput arithmetic for a T4** (65 TFLOPS fp16 peak, realistically ~25–35 % MFU with checkpointing): a 0.5 B model digests roughly **1 M tokens in 2–4 minutes**, i.e. ~**0.5 B tokens/day**. Real knowledge injection wants 10⁹–10¹¹ tokens.
  - ⚠️ **A T4 is a correctness rig for the CPT pipeline, not a knowledge-injection rig.** The code below is the exact code you would run on 8×H100; only `DOMAIN_DOCS`, `BLOCK_SIZE` and the batch size change.
- **`modules_to_save=["embed_tokens"]` (vocabulary extension) is the one thing that *does* blow up memory here:** Qwen2.5-0.5B's embedding matrix is 151,936 × 896 ≈ **136 M params — 4× the entire LoRA budget** — and it is trained densely in fp32. That is why it is behind a flag.

#### Pros & Cons

**Pros**
- **The only paradigm that installs new knowledge and new vocabulary.** Nothing downstream (SFT, DPO, RAG, prompting) substitutes for it.
- **Data is cheap and abundant** — raw unlabelled text. No annotators, no preference pairs, no judge model, no synthetic-data pipeline.
- **100 % token utilisation** — every token in the corpus carries gradient.
- **Compounds with everything after it.** A domain-adapted base is a *better starting point* for SFT and for preference optimisation; the gains multiply rather than compete.
- **Fixes tokenizer economics permanently** — shorter sequences for domain text means cheaper inference forever.

**Cons**
- **Catastrophic forgetting is the default outcome, not an edge case.** Without replay + a sane LR schedule, you trade general capability for domain fluency — and instruction-following/chat ability is usually the first thing to go.
- **Token-hungry to the point of being a budget decision, not a technical one.** Below ~10⁸–10⁹ domain tokens you are mostly buying *style* adaptation, and you should be honest with yourself about that.
- **LoRA/QLoRA is a compromise here specifically.** *Biderman et al., 2024 ("LoRA Learns Less and Forgets Less")* shows low-rank adapters underperform full fine-tuning **most** on CPT-style knowledge acquisition — the same low-rank constraint that limits forgetting also limits learning. Production knowledge injection uses full FT, or high rank (**r ≥ 256**) across all layers **including embeddings**.
- **The output is not usable by end users.** CPT yields a *base* model — a text continuator, not an assistant. It **must** be followed by instruction tuning, so it lengthens the pipeline.
- **Evaluation is indirect.** There is no reward or accuracy signal; you infer success from perplexity deltas and downstream benchmarks, which correlate imperfectly with what you actually want.
- **Data curation becomes the real work** — dedup, quality filtering, PII/licence review, and mixture ratios move the needle more than any hyperparameter here.

#### Metrics to watch (there are no `rewards/*` keys in CPT)

- **`train_loss` / `eval_loss` on held-out *domain* text** — the primary signal; should fall smoothly. A jagged curve means the LR is too high for the re-warm.
- **Domain perplexity ↓ (before vs. after)** — the headline number. Expect a large drop even on a tiny run, because domain register is easy to fit.
- **General perplexity ↑ (before vs. after) — the "forgetting tax".** Always measure it. A few % is a healthy trade; 2× means the replay ratio or LR is wrong.
- **Bits-per-byte (BPB), not perplexity, the moment you extend the vocabulary.** Per-token PPL is *not comparable* across tokenizations — a coarser vocabulary trivially "improves" PPL by predicting fewer, longer tokens. BPB normalises by UTF-8 bytes and is tokenizer-invariant.
- **`grad_norm`** — spikes at the start of the re-warm are expected; persistent spikes mid-run signal LR/data problems.
- **Token utilisation** (packed tokens ÷ tokens forwarded) — should be ~1.0. If not, your packer is broken and you are paying for padding.

---

## 3. Production-Grade Implementation (Colab T4, 16 GB)

**QLoRA (4-bit NF4) + un-masked CLM + EOS-separated sequence packing + 5 % general-text replay, driven by `transformers.Trainer`.**

> ⚙️ **Why plain `Trainer` and not `SFTTrainer` here:** CPT's objective *is* `Trainer`'s default CLM path — `labels == input_ids`, nothing masked. TRL's packing helpers would work, but the whole point of this notebook is that the **data path is the algorithm**, so the packer is written out explicitly instead of hidden behind a flag.

**Executable pipeline:**

| Step | What | CPT-specific detail |
|---|---|---|
| 1 | 4-bit **`Qwen/Qwen2.5-0.5B`** (base!) + high-rank LoRA | **not** `-Instruct` — CPT runs *before* instruction tuning |
| 1.5 | *(optional)* vocabulary extension + mean-init | the part SFT structurally cannot do |
| 2 | `epfl-llm/guidelines` (domain) + `fineweb-edu` (replay) | raw text, one column, no labels |
| 3 | Tokenize **without truncation/padding** → pack into `BLOCK_SIZE` blocks with `eos` separators | ~100 % token utilisation |
| 4 | `TrainingArguments`: **re-warm → cosine re-decay**, 1 epoch, `paged_adamw_8bit` | the Ibrahim et al. recipe |
| 5 | Perplexity + BPB on **domain** *and* **general** held-out sets, before → after | domain gain vs. forgetting tax |
| 6 | Save adapter · export · **continuation** inference (base A/B via `disable_adapter`) | it's a continuator, not a chatbot |

### Environment Setup

In [ ]:
%pip install torchao==0.16.0 transformers datasets peft accelerate bitsandbytes

In [ ]:
import os, gc, math, time
from itertools import islice

import torch
from datasets import load_dataset, Dataset, concatenate_datasets
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments,
    Trainer,
    default_data_collator,
    set_seed,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

set_seed(42)
# Reduce CUDA fragmentation OOMs on the T4 (long packed blocks allocate large contiguous buffers).
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
print("CUDA:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only")

# ---- Hardware-aware dtype selection (do NOT hardcode bf16 on a T4) ----------------
# The Colab T4 is Turing (compute capability 7.5). bfloat16 TENSOR CORES only exist on
# Ampere (SM 8.0) and newer. Forcing bf16 here makes the bitsandbytes 4-bit dequant path
# return garbage, which becomes NaN logits -> NaN loss -> a run that silently learns nothing.
major, _minor = torch.cuda.get_device_capability()
USE_BF16 = major >= 8  # True on A100/L4/H100, False on T4
COMPUTE_DTYPE = torch.bfloat16 if USE_BF16 else torch.float16
print(f"device capability sm_{major}{_minor} | bf16 usable: {USE_BF16} | compute dtype: {COMPUTE_DTYPE}")

# ---- The three knobs that define the SCALE of a CPT run ---------------------------
BLOCK_SIZE   = 1024   # packed context window. 1024 fits a T4 at batch 2; 2048 needs batch 1.
DOMAIN_DOCS  = 300    # ~1.5-2M domain tokens => ~10-20 min on a T4. Production: 1e9-1e11 tokens.
REPLAY_RATIO = 0.05   # 5% general-text replay (Ibrahim et al. 2024 anti-forgetting recipe).

### Step 1 — The **base** checkpoint, 4-bit quantization & the tokenizer

**`Qwen/Qwen2.5-0.5B`, not `Qwen/Qwen2.5-0.5B-Instruct`.** This is the load-bearing decision of the whole notebook:

- CPT sits **before** instruction tuning in the pipeline. Running an un-masked CLM loss over raw documents on an *Instruct* checkpoint actively **destroys** the chat behaviour it was aligned into (raw text contains no chat template, so you are training the model *away* from it).
- There is also no chat template applied anywhere below, and no `add_generation_prompt`. The model's job is **continuation**, not response.

The tokenizer cell also prints how badly the base BPE fragments domain jargon — that fragmentation is the concrete, measurable motivation for Step 1.5.

In [ ]:
# BASE model — CPT precedes instruction tuning. Using an -Instruct checkpoint here would
# train the model away from its own chat template (raw documents contain no chat turns).
base_model_id = "Qwen/Qwen2.5-0.5B"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",  # NF4 is information-theoretically optimal for ~N(0,1) weights
    bnb_4bit_use_double_quant=True,  # quantize the quantization constants too (~0.4 bits/param saved)
    bnb_4bit_compute_dtype=COMPUTE_DTYPE,  # fp16 on T4 (Turing has no bf16 tensor cores)
)

tokenizer = AutoTokenizer.from_pretrained(base_model_id)
if tokenizer.pad_token is None:
    # Packing means we never actually pad during training, but Trainer/collators still want a pad id.
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    base_model_id,
    quantization_config=bnb_config,
    device_map="auto",
    attn_implementation="sdpa",  # memory-efficient attention; FlashAttention-2 needs Ampere+
)
model.config.use_cache = False  # required with gradient checkpointing (KV cache is meaningless in training)

print(f"{model.get_memory_footprint()/1e9:.2f} GB base (4-bit)")
print("tied embeddings:", model.config.tie_word_embeddings,
      "| embedding matrix:", tuple(model.get_input_embeddings().weight.shape))

# ---- The tokenizer-fragmentation problem, measured ------------------------------
# Every extra fragment is extra context, extra FLOPs, and a concept smeared across
# several weakly-related embeddings. This is what vocabulary extension (Step 1.5) fixes,
# and it is structurally impossible to fix with SFT.
JARGON = ["hydroxychloroquine", "thrombocytopenia", "echocardiography",
          "pneumonectomy", "levothyroxine", "immunohistochemistry"]
for w in JARGON:
    pieces = tokenizer.tokenize(w)
    print(f"  {w:<22} -> {len(pieces)} tokens  {pieces}")

### Step 1.5 — *(Optional)* Vocabulary extension with **mean-initialised** embeddings

This is the mechanism the chapter refers to as *"new vocabulary that SFT simply cannot bridge"* — and it is genuinely exclusive to CPT, because it changes the **shape of the model**.

- `tokenizer.add_tokens(...)` → `model.resize_token_embeddings(...)` appends **untrained rows** to the embedding matrix.
- Those rows must be **mean-initialised from the existing embedding matrix** (plus a little noise to break symmetry). Random or zero init puts them far off the learned embedding manifold, which produces garbage logits and a badly-conditioned gradient on step 0.
- New rows only learn if they are **trainable**, which for PEFT means `modules_to_save=["embed_tokens"]` — a **dense fp32 copy of a 136 M-parameter matrix**, ~4× the entire LoRA budget (see [Context Block] → VRAM). Hence the flag.
- ⚠️ **Once the vocabulary changes, per-token perplexity is no longer comparable before vs. after** — different tokenizations produce different token counts. Compare **bits-per-byte** instead (the eval function in Step 5 reports both, for exactly this reason).

Default is **`False`** so the notebook's before/after PPL stays apples-to-apples on a T4. Flip it to `True` to exercise the full path.

In [ ]:
EXTEND_VOCAB = False   # True => add domain tokens AND train the embedding matrix densely (see VRAM note)

# Real domain terms that the base BPE shreds (verified by the printout in Step 1).
NEW_DOMAIN_TOKENS = ["hydroxychloroquine", "thrombocytopenia", "echocardiography",
                     "pneumonectomy", "levothyroxine", "immunohistochemistry"]

modules_to_save = None  # PEFT: extra modules trained DENSELY alongside the LoRA adapters

if EXTEND_VOCAB:
    n_added = tokenizer.add_tokens(NEW_DOMAIN_TOKENS)
    print(f"added {n_added} tokens | tokenizer len -> {len(tokenizer)}")

    with torch.no_grad():
        old_emb = model.get_input_embeddings().weight
        # Mean of the EXISTING rows, computed BEFORE the resize. This is the in-distribution
        # anchor for every new row; random/zero init here is the classic cause of an
        # immediate loss explosion on the first step.
        mean_vec = old_emb.mean(dim=0).clone()
        std = old_emb.std().item() * 0.02  # tiny noise so the new rows aren't perfectly identical

        model.resize_token_embeddings(len(tokenizer))  # appends n_added untrained rows

        new_emb = model.get_input_embeddings().weight
        new_emb[-n_added:] = mean_vec + torch.randn_like(new_emb[-n_added:]) * std
        if not model.config.tie_word_embeddings:
            # Untied models have a separate output matrix that needs the same treatment.
            out = model.get_output_embeddings().weight
            out[-n_added:] = out[:-n_added].mean(dim=0)

    # With TIED embeddings, listing "lm_head" too would silently UN-tie the model and double
    # the dense trainable copy. Only name the module that actually owns the weights.
    modules_to_save = ["embed_tokens"] if model.config.tie_word_embeddings else ["embed_tokens", "lm_head"]

    for w in NEW_DOMAIN_TOKENS[:3]:
        print(f"  {w:<22} -> {tokenizer.tokenize(w)}")   # now a single token
else:
    print("vocabulary extension skipped -> per-token perplexity stays comparable before/after")

### Step 1.6 — LoRA sized **for knowledge injection**, not for style

The adapter config here is deliberately different from the DPO/SimPO notebooks in this repo:

- **`r=64`, `lora_alpha=128`** vs. preference tuning's `r=16`. Low-rank adapters constrain *how much* can be learned — the finding in *Biderman et al., 2024* is that this hurts CPT-style **knowledge acquisition** more than any other regime. Rank is capacity, and CPT needs capacity.
- **All seven linear projections**, including the MLP (`gate/up/down_proj`). Factual associations are stored predominantly in the **FFN** blocks; an attention-only adapter (`q,v` only) is a style adapter, not a knowledge adapter.
- **`use_rslora=True`** — rank-stabilised scaling (`α/√r` instead of `α/r`) keeps update magnitudes sane at higher ranks, which is exactly the regime we just moved into.
- **`lora_dropout=0.05`** — mild; a single pass over a large corpus does not overfit much, so heavy regularisation only slows learning.

In [ ]:
peft_config = LoraConfig(
    r=64,  # CPT needs CAPACITY: rank bounds how much new knowledge can be encoded
    lora_alpha=128,  # alpha = 2r, the standard pairing
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    # ALL linear projections. The MLP trio is not optional here: FFN blocks are where
    # factual key-value associations live, so attention-only adapters learn style, not facts.
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    use_rslora=True,  # alpha/sqrt(r) scaling — stabilises the higher rank
    modules_to_save=modules_to_save,   # None unless Step 1.5 extended the vocabulary
)

# Casts LayerNorms/embeddings to fp32, disables the KV cache, and (crucially) calls
# enable_input_require_grads() so gradients flow through a frozen 4-bit embedding layer
# when gradient checkpointing re-runs the forward pass.
model = prepare_model_for_kbit_training(
    model,
    use_gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},  # reentrant=True breaks with PEFT + checkpointing
)
model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

### Step 2 — The corpora: domain text **+ replay text**

Two loads, one shape. Both datasets are reduced to a **single `text` column** — that is the entire CPT data contract, and it is why the same tokenize/pack code path serves both.

- **`epfl-llm/guidelines`** — Meditron's clinical-guideline corpus (`clean_text`). Some rows are `None`/stubs where a source was excluded for licensing, so they are filtered out.
- **`HuggingFaceFW/fineweb-edu`**, config **`sample-10BT`**, **streamed** — the corpus is TB-scale; `streaming=True` + `islice` pulls only the row-groups we touch. One pass yields two **disjoint** slices: the replay pool and a *never-trained-on* general eval set for measuring the forgetting tax.
- Held-out domain docs are split off **before** packing, so no eval block shares a document with a train block.

In [ ]:
# ---- Domain corpus: raw, unstructured, unlabelled clinical guidelines -------------
raw_domain = load_dataset("epfl-llm/guidelines", split="train")
print("full domain corpus:", raw_domain)

# Shuffle-then-slice a pool FIRST (cheap), then filter inside the pool. Filtering all 38k
# long documents would rewrite ~800MB of Arrow for no benefit.
pool = raw_domain.shuffle(seed=42).select(range(DOMAIN_DOCS + 900))
pool = pool.filter(lambda ex: ex["clean_text"] is not None and len(ex["clean_text"]) > 1000)
assert len(pool) >= DOMAIN_DOCS, f"only {len(pool)} usable docs in the pool; widen the select() above"

domain = (pool.select(range(DOMAIN_DOCS))
              .rename_column("clean_text", "text")
              .select_columns(["text"]))  # drop id/source/title/url: the loss cannot use them

# Hold out whole DOCUMENTS (not blocks) so eval text never leaks into a training block.
domain_split = domain.train_test_split(test_size=10, seed=42)
domain_train, domain_eval = domain_split["train"], domain_split["test"]

chars = sum(len(t) for t in domain_train["text"])
print(f"domain train docs: {len(domain_train)} | eval docs: {len(domain_eval)}")
print(f"domain train text: {chars/1e6:.1f}M chars (~{chars/4/1e6:.1f}M tokens at ~4 chars/token)")
print("\n--- sample domain document (first 400 chars) ---")
print(domain_train[0]["text"][:400])

In [ ]:
# ---- Replay corpus: general text, STREAMED (fineweb-edu is TB-scale) --------------
# Replay is what keeps CPT from lobotomising the base model. It is 5% of the mixture and
# it does more for retention than any hyperparameter in Step 4.
REPLAY_POOL_DOCS = 400
GENERAL_EVAL_DOCS = 60

replay_stream = load_dataset(
    "HuggingFaceFW/fineweb-edu",
    name="sample-10BT",  # the 10B-token sample; still streamed, so only touched shards download
    split="train",
    streaming=True,
)
# ONE pass over the stream, split into two DISJOINT slices: replay pool + never-trained eval.
rows = list(islice(replay_stream, REPLAY_POOL_DOCS + GENERAL_EVAL_DOCS))
replay_pool  = Dataset.from_list([{"text": r["text"]} for r in rows[:REPLAY_POOL_DOCS]])
general_eval = Dataset.from_list([{"text": r["text"]} for r in rows[REPLAY_POOL_DOCS:]])

print(f"replay pool: {len(replay_pool)} docs | general eval (held out): {len(general_eval)} docs")
print("\n--- sample replay document (first 300 chars) ---")
print(replay_pool[0]["text"][:300])

### Step 3 — Tokenize **without truncation or padding**, then **pack**

This is the cell that makes CPT affordable, and it inverts the SFT habit:

- **`truncation=False, padding=False`** — a CPT document is not a training example, it is *material*. Truncating each document to `BLOCK_SIZE` would throw away the majority of a long-form corpus (the printout below quantifies exactly how much).
- **`add_special_tokens=False`** — we control the boundaries ourselves; the packer appends **one `eos` per document** as the explicit document separator.
- **`labels = input_ids`** — the whole of CPT's supervision signal. `Trainer`/`Qwen2ForCausalLM` shift internally, so no manual offset. **Nothing is masked** — every one of the 1024 positions carries gradient, unlike completion-masked SFT.
- The ragged tail of each map batch is dropped — a sub-block remainder per 100 documents, i.e. noise.

In [ ]:
EOS_ID = tokenizer.eos_token_id

def tokenize_fn(batch):
    # No truncation, no padding: the packer owns length control, not the tokenizer.
    # add_special_tokens=False because we append the document separator ourselves.
    # (The "sequence longer than model_max_length" warning below is EXPECTED and harmless —
    #  it is the tokenizer noticing we deliberately did not truncate.)
    return tokenizer(batch["text"], add_special_tokens=False, truncation=False, padding=False)

def pack_fn(batch):
    """Concatenate documents (eos-separated) and slice fixed BLOCK_SIZE windows.

    This is the "wrapped"/concat-and-chunk packing scheme used by essentially every
    pretraining run: token utilisation ~100%, zero padding FLOPs.

    ponytail: naive wrapped packing — tokens CAN attend across the eos boundary to the
    previous document inside a block. Standard practice for pretraining (GPT-3/Llama did
    exactly this) and the eos marker teaches the boundary, but it is a real approximation.
    Upgrade path: emit per-document position_ids + FlashAttention-2 varlen (needs Ampere+)
    or a FlexAttention document block-mask to make boundaries hard.
    """
    buf = []
    for ids in batch["input_ids"]:
        buf.extend(ids)
        buf.append(EOS_ID)  # explicit document boundary
    n = (len(buf) // BLOCK_SIZE) * BLOCK_SIZE   # drop the ragged tail of this map batch
    blocks = [buf[i:i + BLOCK_SIZE] for i in range(0, n, BLOCK_SIZE)]
    return {
        "input_ids": blocks,
        "attention_mask": [[1] * BLOCK_SIZE for _ in blocks],  # all real tokens — nothing to mask
        "labels": [b.copy() for b in blocks],   # CLM: labels == inputs, shifted internally
    }


# ---- Self-check the packer on a hand-verifiable input (BLOCK_SIZE=8, eos=99) ------
# The packer is the one piece of real logic in this notebook; a silent off-by-one here
# would corrupt every label in the run and still train "fine".
# Two synthetic docs: (BLOCK_SIZE-1) + eos + (BLOCK_SIZE+5) + eos = 2*BLOCK_SIZE + 6 tokens.
_probe = pack_fn({"input_ids": [[7] * (BLOCK_SIZE - 1), [8] * (BLOCK_SIZE + 5)]})
assert len(_probe["input_ids"]) == 2, "expected exactly 2 full blocks + a dropped 6-token tail"
assert _probe["input_ids"][0] == [7] * (BLOCK_SIZE - 1) + [EOS_ID]  # eos lands on the boundary
assert _probe["input_ids"][1] == [8] * BLOCK_SIZE  # doc 2 continues into block 2
assert _probe["labels"] == _probe["input_ids"]  # un-masked CLM supervision
assert all(all(m) for m in _probe["attention_mask"])  # zero padding, ever
assert pack_fn({"input_ids": [[7] * (BLOCK_SIZE - 2)]})["input_ids"] == []  # sub-block => no block
print("packer self-check passed")

def tokenize_and_pack(ds, desc):
    tok = ds.map(tokenize_fn, batched=True, batch_size=64,
                 remove_columns=ds.column_names, desc=f"tokenize {desc}")
    packed = tok.map(pack_fn, batched=True, batch_size=100,
                     remove_columns=tok.column_names, desc=f"pack {desc}")
    return tok, packed

domain_tok, domain_blocks = tokenize_and_pack(domain_train, "domain")
_,          replay_blocks = tokenize_and_pack(replay_pool,  "replay")
_,          domain_eval_blocks  = tokenize_and_pack(domain_eval,  "domain-eval")
_,          general_eval_blocks = tokenize_and_pack(general_eval, "general-eval")

print(f"\ndomain blocks: {len(domain_blocks)} x {BLOCK_SIZE} = {len(domain_blocks)*BLOCK_SIZE/1e6:.2f}M tokens")
print(f"replay blocks: {len(replay_blocks)} | domain eval: {len(domain_eval_blocks)} | general eval: {len(general_eval_blocks)}")

# An empty eval set would make Trainer's in-loop evaluation fail with an opaque error.
assert len(domain_eval_blocks) and len(general_eval_blocks), \
    f"eval sets packed to zero blocks — lower BLOCK_SIZE ({BLOCK_SIZE}) or hold out more documents"

In [ ]:
# ---- What packing actually bought us, in numbers --------------------------------
# Compare three ways to feed the SAME corpus through a BLOCK_SIZE window:
#   (a) one document per row, padded to BLOCK_SIZE   -> pays FLOPs for padding
#   (b) one document per row, truncated to BLOCK_SIZE -> silently DISCARDS corpus
#   (c) packed (what we do)                          -> ~100% utilisation, nothing discarded
lens = [len(x) for x in domain_tok["input_ids"]]
real_tokens   = sum(lens)
padded_budget = len(lens) * BLOCK_SIZE  # (a) tokens forwarded
kept_if_trunc = sum(min(l, BLOCK_SIZE) for l in lens)  # (b) tokens actually seen
packed_budget = len(domain_blocks) * BLOCK_SIZE  # (c) tokens forwarded

pad_waste = 100 * max(0.0, 1 - real_tokens / padded_budget)   # 0 when docs are longer than a block
trunc_loss = 100 * (1 - kept_if_trunc / real_tokens)

print(f"documents            : {len(lens)}")
print(f"median doc length    : {sorted(lens)[len(lens)//2]} tokens (block size {BLOCK_SIZE})")
print(f"real corpus tokens   : {real_tokens/1e6:.2f}M")
print()
print(f"(a) pad-to-block     : forwards {padded_budget/1e6:.2f}M tokens -> {pad_waste:.1f}% of FLOPs on PADDING")
print(f"(b) truncate-to-block: sees    {kept_if_trunc/1e6:.2f}M tokens -> {trunc_loss:.1f}% of the corpus DISCARDED")
print(f"(c) packed (ours)    : forwards {packed_budget/1e6:.2f}M tokens -> "
      f"{100*min(1.0, real_tokens/packed_budget):.1f}% utilisation, nothing discarded")
print("\nShort-document corpora lose to (a); long-document corpora lose to (b). Packing beats both.")

In [ ]:
# ---- The training mixture: domain + REPLAY_RATIO of general text -----------------
# Mixing changes the effective objective to (1-a)*L_domain + a*L_general, which keeps a
# gradient component pointing back at the original distribution on every single step.
replay_keep = min(len(replay_blocks), max(1, int(REPLAY_RATIO * len(domain_blocks))))
replay_used = replay_blocks.shuffle(seed=42).select(range(replay_keep))

# Shuffle AFTER concatenating: replay must be interleaved throughout the run, not appended
# as a final phase (a general-text tail at the end of a cosine decay is a different,
# much weaker intervention).
train_ds = concatenate_datasets([domain_blocks, replay_used]).shuffle(seed=42)

print(f"domain blocks : {len(domain_blocks)}")
print(f"replay blocks : {replay_keep}  ({100*replay_keep/len(train_ds):.1f}% of the mixture)")
print(f"train blocks  : {len(train_ds)}  =  {len(train_ds)*BLOCK_SIZE/1e6:.2f}M tokens/epoch")
print("columns       :", train_ds.column_names)

In [ ]:
# Free leftover GPU memory before training.
gc.collect()
torch.cuda.empty_cache()
free, total = torch.cuda.mem_get_info()
print(f"GPU free: {free/1e9:.2f} GB / {total/1e9:.2f} GB")

### Step 4 — `TrainingArguments`: **re-warm → re-decay**, one single epoch

Every non-obvious value here is a CPT decision, not a copy-paste default:

- **`num_train_epochs=1`.** CPT is a *single pass* over a large corpus. Multiple epochs over raw text drives **verbatim memorisation** instead of generalisation — the opposite of what a base model should do. If you need more compute, add tokens, not epochs.
- **`warmup_steps` ≈ 3 %** — the **re-warm**. A base checkpoint arrives at LR ≈ 0 with no optimiser moments; jumping straight to peak LR poisons the Adam moment estimates with the first few high-variance batches.
- **`lr_scheduler_type="cosine"`** — the **re-decay**, annealing into the domain basin.
- **`learning_rate=2e-4`** — a LoRA-scale LR. **Full-fine-tune CPT sits at `1e-5`–`3e-5`**; using a LoRA LR for a full FT is a reliable way to wreck a model.
- **`gradient_accumulation_steps=8`** → effective batch **16 blocks = 16,384 tokens/step**. Pretraining-style objectives want large token batches for gradient-estimate stability; accumulation is how a 16 GB card buys them.
- **`max_grad_norm=1.0`** — the standard pretraining guard against a single bad batch during the fragile re-warm.
- **`optim="paged_adamw_8bit"`** — 8-bit moments, paged to host RAM on spikes.
- **`gradient_checkpointing=True`** — ~30–40 % slower per step, and the reason a 1024-token block fits at all.

In [ ]:
per_device_train_batch_size = 2
gradient_accumulation_steps = 8      # effective batch = 2 * 8 * 1024 = 16,384 tokens/optimizer step
num_train_epochs = 1                 # ONE pass. Repeats memorise raw text verbatim.

# Derive the step count from the real dataset size so warmup scales automatically if you
# change DOMAIN_DOCS / BLOCK_SIZE / batch settings above (world_size=1 on a single T4).
total_optimizer_steps = math.ceil(
    len(train_ds) * num_train_epochs / (per_device_train_batch_size * gradient_accumulation_steps)
)
warmup_steps = max(10, int(0.03 * total_optimizer_steps))   # ~3% RE-WARM
eval_steps = max(20, total_optimizer_steps // 4)

print(f"optimizer steps: {total_optimizer_steps} | warmup: {warmup_steps} | eval every {eval_steps}")
print(f"tokens/optimizer step: {per_device_train_batch_size*gradient_accumulation_steps*BLOCK_SIZE:,}")

In [ ]:
training_args = TrainingArguments(
    output_dir="./cpt_output",
    run_name="cpt-medical-t4",

    # ---- The CPT schedule: RE-WARM then RE-DECAY (Ibrahim et al. 2024) ----
    num_train_epochs=num_train_epochs,   # 1 — single pass over the corpus
    learning_rate=2e-4,  # LoRA-scale. Full FT CPT would be 1e-5..3e-5.
    lr_scheduler_type="cosine",  # re-decay into the domain basin
    warmup_steps=warmup_steps,  # re-warm: rebuild Adam moments at a safe step size
    weight_decay=0.01,
    max_grad_norm=1.0,  # clip — the re-warm phase is where runs blow up

    # ---- Large token batch via accumulation ----
    per_device_train_batch_size=per_device_train_batch_size,
    gradient_accumulation_steps=gradient_accumulation_steps,

    # ---- T4 16 GB hardening ----
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    bf16=USE_BF16, fp16=not USE_BF16,  # match the GPU: fp16 on T4, bf16 on Ampere+
    optim="paged_adamw_8bit",  # 8-bit moments, paged to host RAM on spikes
    dataloader_num_workers=2,

    # ---- Held-out DOMAIN loss during the run ----
    eval_strategy="steps",
    eval_steps=eval_steps,
    per_device_eval_batch_size=2,

    logging_steps=10,
    save_strategy="no",  # we save the adapter explicitly after training
    report_to="none",
    label_names=["labels"],  # silences the PEFT + Trainer label-detection warning
    seed=42,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=domain_eval_blocks,  # held-out DOMAIN blocks: the primary CPT signal
    data_collator=default_data_collator, # blocks are already fixed-length with labels — just stack them
    processing_class=tokenizer,
)

### Step 5 — **Baseline** perplexity, before a single gradient step

CPT has no reward and no accuracy to watch, so the *only* honest way to report it is a **before/after pair on two held-out sets**:

- **Domain perplexity** — did we learn the domain? (expected: large drop)
- **General perplexity** — what did we forget? (expected: small rise = the *forgetting tax*)

Reporting the first without the second is how models quietly get lobotomised in production.

The function also returns **bits-per-byte**, which normalises by UTF-8 bytes instead of tokens and is therefore **tokenizer-invariant** — the metric you *must* switch to if `EXTEND_VOCAB=True`.

> LoRA's `B` matrices are zero-initialised, so the adapter is an exact **identity at step 0** — measuring now measures the untouched base model. After training, `with model.disable_adapter():` reproduces this baseline at any time, with no second model in memory.

In [ ]:
@torch.no_grad()
def eval_ppl_bpb(model, ds, batch_size=2, max_blocks=40):
    """Token-level perplexity AND tokenizer-invariant bits-per-byte over packed blocks."""
    was_training = model.training
    model.eval()
    nll_sum, n_tokens, n_bytes = 0.0, 0, 0

    for i in range(0, min(len(ds), max_blocks), batch_size):
        rows = ds[i:i + batch_size]
        ids = torch.tensor(rows["input_ids"], device=model.device)
        out = model(input_ids=ids, attention_mask=torch.ones_like(ids), labels=ids)

        # HF shifts labels internally, so each row of length L supplies L-1 predictions and
        # out.loss is the MEAN nll over exactly those. Re-weight to sum before accumulating.
        preds = ids.numel() - ids.shape[0]
        nll_sum += out.loss.float().item() * preds
        n_tokens += preds
        # UTF-8 bytes of the same text (off by the 1 unpredicted token per row: ~0.1%).
        n_bytes += sum(len(t.encode("utf-8")) for t in tokenizer.batch_decode(ids))

    if was_training:
        model.train()
    mean_nll = nll_sum / n_tokens
    return {
        "ppl": math.exp(mean_nll),  # per-token; NOT comparable across tokenizers
        "bpb": mean_nll / math.log(2) * n_tokens / n_bytes,  # per-byte; tokenizer-invariant
        "tokens": n_tokens,
    }

# Zero-init LoRA B => the adapter is the identity right now => this IS the base model.
before_domain  = eval_ppl_bpb(model, domain_eval_blocks)
before_general = eval_ppl_bpb(model, general_eval_blocks)

print(f"BASE  domain  : ppl {before_domain['ppl']:8.3f} | bpb {before_domain['bpb']:.4f}")
print(f"BASE  general : ppl {before_general['ppl']:8.3f} | bpb {before_general['bpb']:.4f}")

### Step 6 — Train

Watch **`loss` fall smoothly** and **`grad_norm` settle after the warmup**. There are no reward metrics to read here — a jagged loss curve after warmup means the learning rate is too high for the re-warm you configured.

In [ ]:
torch.cuda.reset_peak_memory_stats()
t0 = time.time()

train_result = trainer.train()

wall_min = (time.time() - t0) / 60
peak_gb = torch.cuda.max_memory_allocated() / 1e9
tokens_seen = len(train_ds) * BLOCK_SIZE * num_train_epochs

print(f"\nwall clock: {wall_min:.1f} min")
print(f"peak VRAM: {peak_gb:.2f} GB")
print(f"tokens seen: {tokens_seen/1e6:.2f}M")
print(f"throughput: {tokens_seen/(wall_min*60):,.0f} tokens/sec")
print(f"final train loss: {train_result.training_loss:.4f}")

# Save the domain-adapted LoRA adapter (a few MB, not GB). The tokenizer goes with it —
# mandatory if EXTEND_VOCAB added rows, since the adapter's embedding matrix depends on it.
trainer.save_model("./cpt_domain_adapter")
tokenizer.save_pretrained("./cpt_domain_adapter")
print("Saved -> ./cpt_domain_adapter")

In [ ]:
# ---- The two numbers that define whether this CPT run was any good ---------------
after_domain  = eval_ppl_bpb(model, domain_eval_blocks)
after_general = eval_ppl_bpb(model, general_eval_blocks)

def pct(a, b):
    return 100 * (b - a) / a

print(f"{'set':<10}{'metric':<8}{'before':>12}{'after':>12}{'change':>12}")
print("-" * 54)
for name, b, a in [("domain", before_domain, after_domain), ("general", before_general, after_general)]:
    for m in ("ppl", "bpb"):
        print(f"{name:<10}{m:<8}{b[m]:>12.4f}{a[m]:>12.4f}{pct(b[m], a[m]):>11.1f}%")

print(f"\nDOMAIN GAIN: {-pct(before_domain['ppl'], after_domain['ppl']):.1f}% perplexity reduction")
print(f"FORGETTING TAX: {pct(before_general['ppl'], after_general['ppl']):+.1f}% general perplexity")
if EXTEND_VOCAB:
    print("NOTE: vocabulary was extended -> read the BPB rows, NOT ppl (tokenizations differ).")
if pct(before_general["ppl"], after_general["ppl"]) > 25:
    print("WARNING: >25% general regression. Raise REPLAY_RATIO or lower the learning rate.")

---

## **[Key Observations]**

*Fill in from the runs above — CPT is judged on a before/after pair, never on a single number.*

### Run configuration

| Setting | Value |
|---|---|
| Base model | `Qwen/Qwen2.5-0.5B` (base) |
| `BLOCK_SIZE` | |
| `DOMAIN_DOCS` / domain tokens | |
| `REPLAY_RATIO` (actual %) | |
| LoRA `r` / `alpha` / targets | |
| `EXTEND_VOCAB` | |
| LR / warmup steps / scheduler | |
| Effective batch (tokens/step) | |

### Results

| Metric | Before | After | Δ | Expected direction |
|---|---|---|---|---|
| **Domain** perplexity | | | | **↓ strongly** |
| **Domain** bits-per-byte | | | | ↓ |
| **General** perplexity | | | | ↑ *slightly* (forgetting tax) |
| **General** bits-per-byte | | | | ↑ slightly |
| Final `train_loss` | | | | — |
| Final `eval_loss` (domain) | | | | ↓ |

### Efficiency & hardware

| Metric | Value |
|---|---|
| Token utilisation from packing (%) | |
| Corpus discarded by naive truncation (%) | |
| Peak VRAM (GB) | |
| Wall clock (min) | |
| Throughput (tokens/sec) | |
| Trainable params (% of total) | |

### Qualitative

- Does the model continue a clinical sentence in **domain register** (guideline phrasing, evidence grades, drug names) vs. the base model's generic continuation?
- Does it still produce coherent **general-domain** text, or has the replay ratio proven too small?
- If `EXTEND_VOCAB=True`: do the new single-token terms appear in generations, and did the loss spike at step 0?

### Things worth logging every time

- `grad_norm` behaviour across the warmup boundary (spike → settle is healthy).
- Whether the loss curve is smooth after warmup — jaggedness ⇒ LR too high.
- Sensitivity sweep: `REPLAY_RATIO ∈ {0, 0.01, 0.05, 0.15}` against the forgetting tax; `0` is the instructive control.

## Export — Download the Domain-Adapted Adapter (Optional)

In [ ]:
import shutil, os

folder_to_zip = "./cpt_domain_adapter"
output_filename = "cpt_domain_adapter.zip"

shutil.make_archive(output_filename.replace(".zip", ""), "zip", folder_to_zip)
if os.path.exists(output_filename):
    print(f"File: {output_filename}  ({os.path.getsize(output_filename)/1e6:.2f} MB)")
else:
    print("Zip not found — run training + save first.")

### Download to your machine

In [ ]:
from google.colab import files
files.download(output_filename)

### Or back up to Google Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

destination_folder = "/content/drive/MyDrive/colab_models"
if os.path.exists(output_filename):
    os.makedirs(destination_folder, exist_ok=True)
    shutil.copy(output_filename, os.path.join(destination_folder, output_filename))
    print("Backed up to Drive:", destination_folder)
else:
    print("Error: zip not found. Did training + zipping finish?")

---

## Model Usage — **Continuation**, not chat

A CPT'd model is still a **base model**: a text continuator. There is no chat template, no `add_generation_prompt`, and asking it a question gets you a plausible *continuation of the question*, not an answer. That is not a bug — instruction tuning is the **next** stage of the pipeline.

The A/B below is free: `PeftModel` keeps the base weights intact, so **`with model.disable_adapter():`** yields the un-adapted base output in the same process, with no second model in VRAM.

In [ ]:
# IMPORTANT: free the TRAINING model before loading a second copy for inference. The trainer
# and its optimizer state are still resident; building another 4-bit model on top of them
# is what tips a 16 GB T4 over.
import gc, torch

for _obj in ["trainer", "model"]:
    if _obj in globals():
        del globals()[_obj]
gc.collect()
torch.cuda.empty_cache()
free, total = torch.cuda.mem_get_info()
print(f"GPU free before inference load: {free/1e9:.2f} GB / {total/1e9:.2f} GB")

from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

# Re-derive the dtype here so this cell works standalone after a runtime restart.
major, _minor = torch.cuda.get_device_capability()
USE_BF16 = major >= 8
COMPUTE_DTYPE = torch.bfloat16 if USE_BF16 else torch.float16

base_model_id = "Qwen/Qwen2.5-0.5B"
adapter_path = "./cpt_domain_adapter"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=COMPUTE_DTYPE,
)

tokenizer = AutoTokenizer.from_pretrained(adapter_path)  # from the ADAPTER: may carry added tokens
base = AutoModelForCausalLM.from_pretrained(
    base_model_id,
    quantization_config=bnb_config,
    device_map="auto",
    attn_implementation="sdpa",
)

# If Step 1.5 extended the vocabulary, the base MUST be resized before the adapter's
# embedding matrix can be loaded into it — PEFT will not do this for you.
if len(tokenizer) != base.get_input_embeddings().weight.shape[0]:
    print(f"resizing embeddings {base.get_input_embeddings().weight.shape[0]} -> {len(tokenizer)}")
    base.resize_token_embeddings(len(tokenizer))

model = PeftModel.from_pretrained(base, adapter_path)
model.config.use_cache = True   # re-enable the KV cache for generation (disabled during training)
model.eval()

print("embedding rows:", model.get_input_embeddings().weight.shape[0], "| tokenizer len:", len(tokenizer))

In [ ]:
@torch.no_grad()
def continue_text(prefix, max_new_tokens=100, use_adapter=True):
    """Raw continuation — NO chat template. A CPT'd base model completes text, it does not answer."""
    inputs = tokenizer(prefix, return_tensors="pt", add_special_tokens=False)
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    def _gen():
        probe = model(**inputs).logits
        if not torch.isfinite(probe).all():
            raise RuntimeError(
                "Non-finite logits — almost always a dtype/hardware mismatch (bf16 on a Turing T4) "
                "or a diverged run. Check COMPUTE_DTYPE and the loss curve."
            )
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,  # greedy: a fair A/B needs determinism, not sampling noise
            repetition_penalty=1.1,
            pad_token_id=tokenizer.pad_token_id or tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
        return tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

    if use_adapter:
        return _gen()
    with model.disable_adapter():  # base weights only — the free A/B, no second model in VRAM
        return _gen()


prefixes = [
    "In patients with newly diagnosed type 2 diabetes, first-line pharmacological therapy is",
    "Recommendation: for suspected pulmonary embolism, the initial diagnostic imaging modality should be",
]

for i, p in enumerate(prefixes, 1):
    print(f"\n[Prefix {i}] {p}")
    print(f"  [BASE ] ...{continue_text(p, use_adapter=False).strip()}")
    print(f"  [CPT  ] ...{continue_text(p, use_adapter=True).strip()}")
    print("-" * 70)